# Human Activity Recognition with CNN, SMOTE, and PCA

本 Notebook 演示如何在人体活动识别项目中，将 SMOTE（平衡数据）和 PCA（降维）集成到 CNN 流程中。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization


In [ ]:
# 加载数据
X_train_raw = np.loadtxt('Train/X_train.txt')
y_train_raw = np.loadtxt('Train/y_train.txt').astype(int) - 1  # 将标签转换为 0 索引

X_test = np.loadtxt('Test/X_test.txt')
y_test = np.loadtxt('Test/y_test.txt').astype(int) - 1


In [ ]:
# 使用 SMOTE 对训练数据进行平衡
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_raw, y_train_raw)

# 使用 PCA 降维到 100 维（可根据需要调整 n_components）
pca = PCA(n_components=100)
X_train_pca = pca.fit_transform(X_train_balanced)
X_test_pca = pca.transform(X_test)


In [ ]:
# 重塑数据为 CNN 格式：(样本数, 特征数, 1)
X_train = X_train_pca.reshape((X_train_pca.shape[0], X_train_pca.shape[1], 1))
X_test = X_test_pca.reshape((X_test_pca.shape[0], X_test_pca.shape[1], 1))


In [ ]:
# 构建 CNN 模型
model = Sequential([
    Conv1D(64, 3, activation='relu', input_shape=(X_train.shape[1], 1)),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),

    Conv1D(128, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(12, activation='softmax')  # 12 个活动类别
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# 训练模型
history = model.fit(X_train, y_train_balanced, epochs=30, batch_size=64, validation_split=0.2)

In [ ]:
# 评估模型
loss, acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {acc:.4f}")

# 绘制混淆矩阵
y_pred = np.argmax(model.predict(X_test), axis=1)
conf_mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 8))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# 输出分类报告
print("\nClassification Report:")
print(classification_report(y_test, y_pred))